In [1]:
%pip install tensorflow tensorflow-hub numpy matplotlib ipython scipy setuptools

  Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached tensorflow_hub-0.16.1-py2.py3-none-any.whl (30 kB)

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
%pip install youtube-dl


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import csv
import scipy

import matplotlib.pyplot as plt
from IPython.display import Audio
from scipy.io import wavfile

2025-08-07 15:46:02.195376: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-07 15:46:02.195446: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-07 15:46:02.197578: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-07 15:46:02.216979: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-07 15:46:03.184571: W tensorflow/compiler/tf2

In [3]:
model = hub.load('https://tfhub.dev/google/yamnet/1')

2025-08-07 15:46:08.447307: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-08-07 15:46:08.767731: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [4]:
def class_names_from_csv(class_map_csv_text):
    class_names = []
    with tf.io.gfile.GFile(class_map_csv_text) as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            class_names.append(row['display_name'])
    return class_names

class_map_path = model.class_map_path().numpy()
class_names = class_names_from_csv(class_map_path)

In [5]:
def ensure_sample_rate(original_sample_rate, waveform, desired_sample_rate=16000):
    if original_sample_rate != desired_sample_rate:
        desired_length = int(round(float(len(waveform)) / original_sample_rate * desired_sample_rate))
        waveform = scipy.signal.resample(waveform, desired_length)
    return desired_sample_rate, waveform

In [13]:
wav_file_name = './_DIJYqGsR1jB.wav'
sample_rate, wav_data = wavfile.read(wav_file_name, 'rb')
sample_rate, wav_data = ensure_sample_rate(sample_rate, wav_data)

duration = len(wav_data) * 1/sample_rate
print(f'Sample rate: {sample_rate} Hz')
print(f'Total duration: {duration:.2f} seconds')
print(f'size of wav_data: {len(wav_data)}')

Audio(wav_data, rate=sample_rate)

Sample rate: 16000 Hz
Total duration: 44.14 seconds
size of wav_data: 706281


error: ushort format requires 0 <= number <= 65535

In [9]:
waveform = wav_data / tf.int16.max

In [10]:
scores, embeddings, spectrogram = model(waveform)

In [11]:
scores_np = scores.numpy()
spectrogram_np = spectrogram.numpy()
infered_class = class_names[scores_np.mean(axis=0).argmax()]
print(f'The main sound is: {infered_class}')

The main sound is: Animal


In [10]:
import yt_dlp as youtube_dl

In [11]:
link = 'https://www.instagram.com/reel/DIJYqGsR1jB/?igsh=MWJnaXl1d3lkMHdrMQ=='
ydl_opts = {
'format': 'bestaudio/best',
'postprocessors': [{
'key': 'FFmpegExtractAudio',
'preferredcodec': 'wav',
'preferredquality': '192',
}],
'outtmpl': '_%(id)s.%(ext)s',
}
with youtube_dl.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([link])

[Instagram] Extracting URL: https://www.instagram.com/reel/DIJYqGsR1jB/?igsh=MWJnaXl1d3lkMHdrMQ==
[Instagram] DIJYqGsR1jB: Setting up session
[Instagram] DIJYqGsR1jB: Downloading JSON metadata
[info] DIJYqGsR1jB: Downloading 1 format(s): dash-561366986984252ad
[download] Destination: _DIJYqGsR1jB.m4a
[download] 100% of  301.79KiB in 00:00:00 at 1.62MiB/s     
[FixupM4a] Correcting container of "_DIJYqGsR1jB.m4a"
[ExtractAudio] Destination: _DIJYqGsR1jB.wav
Deleting original file _DIJYqGsR1jB.m4a (pass -k to keep)
